# HR Attrition Predictive Model

## Project Overview
The objective of this notebook is to build a machine learning model capable of predicting employee attrition based on the IBM HR dataset. By identifying key factors that contribute to an employee's decision to leave, we can generate actionable insights and risk scores for the HR department.

## Step 1: Environment Setup and Data Loading
Importing necessary libraries for data manipulation and loading the raw dataset into a pandas DataFrame.

In [1]:
import pandas as pd
import numpy as np

# Load the IBM HR Attrition dataset
df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')

# Display the first 5 rows to verify successful loading
display(df.head())

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


## Step 2: Data Preprocessing & Feature Engineering
Machine learning models require numerical input. In this step, we prepare the dataset by:
1. Removing irrelevant columns that lack predictive value (e.g., unique IDs or constants).
2. Encoding the target variable (`Attrition`) into a binary format (1 for 'Yes', 0 for 'No').
3. Applying One-Hot Encoding to categorical features (e.g., Department, JobRole, Gender) to convert them into numerical representations.

In [2]:
# 1. Drop columns that are identical for all employees or serve only as identifiers
columns_to_drop = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
df = df.drop(columns=columns_to_drop)

# 2. Convert the target variable to binary
df['Attrition'] = df['Attrition'].apply(lambda x: 1 if x == 'Yes' else 0)

# 3. Convert all remaining categorical variables into dummy/indicator variables
df_model = pd.get_dummies(df, drop_first=True)

# Verify the transformation
print(f"Original dataset shape: {df.shape}")
print(f"Processed dataset shape: {df_model.shape}")
display(df_model.head())

Original dataset shape: (1470, 31)
Processed dataset shape: (1470, 45)


,Age,Attrition,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,...,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single,OverTime_Yes
0,41,1,1102,1,2,2,94,3,2,4,...,False,False,False,False,False,True,False,False,True,True
1,49,0,279,8,1,3,61,2,2,2,...,False,False,False,False,True,False,False,True,False,False
2,37,1,1373,2,2,4,92,2,1,3,...,True,False,False,False,False,False,False,False,True,True
3,33,0,1392,3,4,4,56,3,1,3,...,False,False,False,False,True,False,False,True,False,True
4,27,0,591,2,1,1,40,3,1,2,...,True,False,False,False,False,False,False,True,False,False


### Step 3: Feature Encoding and Train-Test Split
Machine learning algorithms require numerical input. We will use One-Hot Encoding to convert the remaining categorical variables (e.g., Department, BusinessTravel, JobRole) into binary indicator columns. Afterward, we separate the features (`X`) from our target variable (`y`) and split the dataset into training (80%) and testing (20%) sets. The `stratify` parameter ensures the class distribution is maintained across both sets.

In [3]:
from sklearn.model_selection import train_test_split

# Convert categorical variables into dummy/indicator variables
df_encoded = pd.get_dummies(df, drop_first=True)

# Separate features (X) and target (y)
X = df_encoded.drop('Attrition', axis=1)
y = df_encoded['Attrition']

# Split the data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Verify the splits and the new number of columns after encoding
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

Training features shape: (1176, 44)
Testing features shape: (294, 44)


### Step 4 (Revised): Handling Class Imbalance and Custom Decision Thresholds
The initial Random Forest model exhibited a critical flaw: a severe bias toward the majority class (retained employees), resulting in an unacceptable recall for actual attrition cases. To rectify this, we implement the Synthetic Minority Over-sampling Technique (SMOTE). By mathematically generating synthetic profiles of departing employees, we provide the algorithm with a perfectly balanced dataset during the training phase. Crucially, SMOTE is applied *only* to the training data to prevent data leakage.

Furthermore, relying on the default 0.5 (50%) classification threshold is inefficient for HR risk management. We will manually lower the decision threshold to 0.30 to prioritize "Recall" (identifying flight risks early), accepting a slight drop in precision as a standard business trade-off.

In [6]:
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Apply SMOTE exclusively to the training data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# 2. Train the Random Forest on the balanced SMOTE dataset
rf_model_smote = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model_smote.fit(X_train_smote, y_train_smote)

# 3. Extract exact probabilities instead of strict 0/1 predictions
y_pred_probs = rf_model_smote.predict_proba(X_test)[:, 1]

# 4. Apply a Custom Decision Threshold (e.g., 0.30 instead of default 0.50)
custom_threshold = 0.30
y_pred_custom = (y_pred_probs >= custom_threshold).astype(int)

# 5. Evaluate the newly calibrated predictions
print(f"Model Accuracy (Threshold {custom_threshold}): {accuracy_score(y_test, y_pred_custom):.4f}\n")
print("Calibrated Classification Report:")
print(classification_report(y_test, y_pred_custom))

Model Accuracy (Threshold 0.3): 0.7347

Calibrated Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.74      0.83       247
           1       0.34      0.68      0.45        47

    accuracy                           0.73       294
   macro avg       0.63      0.71      0.64       294
weighted avg       0.83      0.73      0.77       294



### Step 5: Actionable Insights and Excel Export
A classification model is only as valuable as the business decisions it drives. Having calibrated our threshold to prioritize identifying flight risks, we now extract the specific probability scores for each employee profile. We filter for high-risk individuals (attrition probability >= 30%) and export this actionable list to an Excel report, enabling the HR department to proactively conduct targeted retention interviews.

In [7]:
# Extract the exact probability of belonging to class 1 (Attrition = Yes)
risk_probabilities = rf_model_smote.predict_proba(X_test)[:, 1]

# Create a copy of the test set to build our final HR report
risk_report = X_test.copy()
risk_report['Attrition_Risk_Score'] = risk_probabilities

# Filter for high-risk employees (>= 30% probability) and sort descending
high_risk_list = risk_report[risk_report['Attrition_Risk_Score'] >= custom_threshold].sort_values(
    by='Attrition_Risk_Score', ascending=False
)

# Export the final actionable report to Excel
high_risk_list.to_excel("HR_High_Risk_Employees_Report.xlsx", index=False)
print(f"Report successfully exported! Identified {len(high_risk_list)} high-risk profiles.")

# Display the top 5 most vulnerable employee profiles for a quick preview
display(high_risk_list.head())

Report successfully exported! Identified 95 high-risk profiles.


,Age,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,...,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single,OverTime_Yes,Attrition_Risk_Score
301,18,812,10,3,4,69,2,1,3,1200,...,False,False,False,False,False,True,False,True,False,0.85
688,19,419,21,3,4,37,2,1,2,2121,...,False,False,False,False,False,True,False,True,True,0.79
1061,24,830,13,2,4,78,3,1,2,2033,...,False,False,False,False,False,True,True,False,False,0.76
911,25,599,24,1,3,73,1,1,4,1118,...,False,False,False,False,False,True,False,True,True,0.76
357,21,756,1,1,1,99,2,1,2,2174,...,False,False,False,False,False,True,False,True,True,0.76
